# Frame Zipper

Zips reconstructed frames from the `ami-e2vid-pipeline` kernel output into a single file.

**Setup:** Add Data → Code output → `gennepy/ami-e2vid-pipeline`  
**Run:** Run All → download `frames_e2vid.zip` from the Output tab (~2.5 GB)


In [ ]:
import subprocess
from pathlib import Path

# Show everything mounted under /kaggle/input/
print('=== /kaggle/input/ contents ===')
for p in sorted(Path('/kaggle/input').iterdir()):
    print(' ', p)

print()

# Find frames anywhere under /kaggle/input/
result = subprocess.run(
    ['find', '/kaggle/input', '-maxdepth', '7',
     '(', '-name', 'frame_000000.jpg', '-o', '-name', 'frame_000000.png',
          '-o', '-name', 'timestamps.txt', ')'],
    capture_output=True, text=True
)
print('=== Frame/timestamp locations ===')
print(result.stdout[:3000] or '(nothing found)')


In [ ]:
import zipfile

# Set INPUT_FRAMES to the directory containing the sequence_* folders
# (adjust based on debug output above)
INPUT_FRAMES = INPUT   # e.g. INPUT / 'data' / 'processed'  if that's what debug shows

OUT = Path('/kaggle/working/frames_e2vid.zip')
SEQUENCES = ['sequence_84', 'sequence_85', 'sequence_201', 'sequence_127']

frame_count = 0
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED, compresslevel=1) as zf:
    for seq in SEQUENCES:
        recon_dir = INPUT_FRAMES / seq / 'reconstruction_e2vid'
        if not recon_dir.exists():
            # fallback: try data/processed/ prefix
            recon_dir = INPUT_FRAMES / 'data' / 'processed' / seq / 'reconstruction_e2vid'
        ts_file = recon_dir / 'timestamps.txt'
        if ts_file.exists():
            zf.write(ts_file, f'{seq}/timestamps.txt')
        frames = sorted(recon_dir.glob('frame_*.jpg')) or sorted(recon_dir.glob('frame_*.png'))
        for frame in frames:
            zf.write(frame, f'{seq}/{frame.name}')
            frame_count += 1
        print(f'{seq}: {len(frames)} frames  (recon_dir={recon_dir})')

size_mb = OUT.stat().st_size / 1e6
print(f'\nframes_e2vid.zip: {frame_count} frames, {size_mb:.0f} MB → Output tab')
